In [ ]:
# Parámetros que serán recibidos desde el pipeline de Fabric
source_layer = f"lh_bronze_qad1"
source_schema = f"QAD"
source_table = f"PT_MSTR"

target_layer = f"lh_silver_erp"
target_schema = f"md"
target_table = f"materials"

# Nombre de la tabla de control
control_table = "bronze_to_silver_control"

# Clave(s) primaria(s) para el MERGE, separadas por coma si son compuestas (de target)
#primary_keys = "PROGRESS_RECID, PT_DOMAIN"
primary_keys = "record_id, company_code"

# Columna para la carga incremental (de source)
watermark_column = "PROGRESS_RECID"
last_watermark_value = None
#created_at_column = None
#last_created_at_value = None
updated_at_column = "PT_MOD_DATE"
last_updated_at_value = None

# Mapeo de columnas en formato JSON string
column_mapping_json = """
[{"source":"PT_PART","target":"product_id","cast":"STRING"},{"source":"PT_DESC1","target":"description_1","cast":"STRING","transform":"title_case"},{"source":"PT_DESC2","target":"description_2","cast":"STRING","transform":"title_case"},{"source":"UPPER(PT_UM)","target":"unit_of_measure","cast":"STRING"},{"source":"COALESCE(UPPER(PT_DRAW), ' ')","target":"drawing_number","cast":"STRING"},{"source":"COALESCE(UPPER(PT_PROMO), ' ')","target":"promo_number","cast":"STRING"},{"source":"COALESCE(UPPER(PT_PROD_LINE), ' ')","target":"product_line","cast":"STRING"},{"source":"COALESCE(UPPER(PT_GROUP), ' ')","target":"product_group","cast":"STRING"},{"source":"COALESCE(UPPER(PT_PART_TYPE), ' ')","target":"part_type","cast":"STRING"},{"source":"UPPER(PT_STATUS)","target":"status","cast":"STRING"},{"source":"UPPER(PT_ABC)","target":"abc_class","cast":"STRING"},{"source":"UPPER(PT_LOC)","target":"location","cast":"STRING"},{"source":"PT_ABC_AMT","target":"abc_amount","cast":"DECIMAL(18,2)"},{"source":"PT_AVG_INT","target":"average_interval","cast":"INT"},{"source":"PT_CYC_INT","target":"cycle_interval","cast":"INT"},{"source":"COALESCE(PT_BUYER, ' ')","target":"buyer","cast":"STRING"},{"source":"COALESCE(PT_VEND, ' ')","target":"vendor_id","cast":"STRING"},{"source":"PT_PRICE","target":"price","cast":"DECIMAL(18,2)"},{"source":"PT_XTOT_STD","target":"standard_total_cost","cast":"DECIMAL(18,2)"},{"source":"PT_STD_DATE","target":"standard_cost_date","cast":"DATE"},{"source":"PT_REV","target":"revision","cast":"STRING"},{"source":"PT_ADDED","target":"date_added","cast":"DATE"},{"source":"COALESCE(UPPER(PT_SITE), ' ')","target":"site","cast":"STRING"},{"source":"PT_MOD_DATE","target":"modification_date","cast":"DATE"},{"source":"COALESCE(LOWER(PT_USERID), ' ')","target":"user_id","cast":"STRING"},{"source":"PT_DOMAIN","target":"company_code","cast":"STRING"},{"source":"PROGRESS_RECID","target":"record_id","cast":"INT"},{"source":"current_timestamp()","target":"last_updated_at","cast":"TIMESTAMP"}]
"""
control_table = "bronze_to_silver_control"

last_run_at = None
last_success_run_at = None

In [ ]:
# Importaciones y Configuración 
from pyspark.sql.functions import max, col, lit, current_date, date_sub 
from delta.tables import DeltaTable
from pyspark.sql.functions import udf, expr, col
from pyspark.sql.types import StringType
from pyspark.sql import functions as F, types as T

import notebookutils
import json
import sys
import time
import random

# Lista de preposiciones y artículos a mantener en minúsculas
LOWERCASE_WORDS = {'a', 'ante', 'bajo', 'cabe', 'con', 'contra', 'de', 'desde', 'en', 'del',
                   'entre', 'hacia', 'hasta', 'para', 'por', 'según', 'sin', 'so', 
                   'sobre', 'tras', 'y', 'e', 'ni', 'que', 'el', 'la', 'los', 'las', 'un', 'una',
                   'S.' 'SA', 'S.A.', 'S.A', 'CV', 'C.V.','C.V','MAB','ZDI','LLC','USA','R.L.'}

# Configuración para LEER fechas antiguas de fuentes Parquet/Delta
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")

# Configuración para ESCRIBIR fechas antiguas a destinos Parquet/Delta (RECOMENDADA)
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

audit_table = f"UPDT_LOG"

# ───────── Rutas absolutas (para Fabric) ────────────────
source_table_name = f"{source_layer}.{source_schema}.{source_table}"
target_table_name = f"{target_layer}.{target_schema}.{target_table}"
control_table_name = f"lh_control_erp.dbo.{control_table}"
audit_table_name = f"{source_layer}.{source_schema}.{audit_table}"

if updated_at_column == "":
    updated_at_column = None

# Imprimir los parámetros recibidos para facilitar la depuración en las ejecuciones del pipeline
print("--- Parámetros de Ejecución ---")
print(f"Tabla Origen (Bronze): {source_table_name}")
print(f"Tabla Destino (Silver): {target_table_name}")
print(f"Tabla de Control: {control_table_name}")
print(f"Claves Primarias: {primary_keys}")
print(f"Columna Rec Id: {watermark_column}")
print("-----------------------------")

In [ ]:
# ───────── Control de concurrencia para actualización de tabla de control ─────────
MAX_RETRIES = 10
BASE_DELAY_SEC = 3
MAX_DELAY_SEC = 40

def is_concurrent_error(ex: Exception) -> bool:
    """Detecta si el error es por concurrencia en Delta Lake."""
    msg = str(ex).lower()
    return "concurrentappendexception" in msg or ("concurrent" in msg and "delta" in msg)

def run_control_update(update_query: str) -> None:
    """Ejecuta un UPDATE sobre la tabla de control con reintentos ante concurrencia."""
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            spark.sql(update_query)
            if attempt > 1:
                print(f"   Tabla de control actualizada (intento {attempt}).")
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                raise
    if last_error is not None:
        raise last_error

def run_delta_operation_with_retry(fn, operation_name: str = "operación Delta") -> None:
    """Ejecuta una operación Delta (MERGE, etc.) con reintentos ante ConcurrentAppendException."""
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            fn()
            if attempt > 1:
                print(f"   {operation_name} completada (intento {attempt}).")
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada en {operation_name} (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                raise
    if last_error is not None:
        raise last_error

# ───────── Función utilitaria ───────────────────────────
def delta_table_exists(path: str) -> bool:
    try:
        DeltaTable.forPath(spark, path)
        return True
    except AnalysisException:
        return False
        
def to_custom_title_case(text):
    """
    Convierte un string a tipo título, excepto ciertas preposiciones y artículos.
    La primera palabra siempre se capitaliza.
    """
    if text is None:
        return None
    
    words = text.lower().split()
    title_cased_words = [word.capitalize() if word not in LOWERCASE_WORDS else word for word in words]
    
    if title_cased_words:
        title_cased_words[0] = title_cased_words[0].capitalize()
        
    return " ".join(title_cased_words)

# Registra la función como una UDF de Spark para poder usarla en transformaciones
custom_title_case_udf = udf(to_custom_title_case, StringType())

def apply_company_code_fix(
    df,
    col_name="company_code",
    mapping=None,
    normalize=True,
    coerce_non_string=False,
    verbose=True
):
    """
    Normaliza company_code:
      ST -> STU
      PS -> PRO
    - Idempotente: si no existe la columna, no hace nada.
    - Usa map lookup con sintaxis moderna: repl[key_expr]
    """
    # localizar la columna real (case-insensitive)
    real_col = next((c for c in df.columns if c.lower() == col_name.lower()), None)
    if real_col is None:
        if verbose:
            print(f"[apply_company_code_fix] Columna '{col_name}' no existe. Sin cambios.")
        return df

    # mapeo base (+ overrides opcionales)
    base_map = {"ST": "STU", "PS": "PRO"}
    if mapping:
        base_map.update({str(k).upper(): str(v) for k, v in mapping.items()})

    # construir el map dinamicamente
    kv = []
    for k, v in base_map.items():
        kv += [F.lit(k), F.lit(v)]
    repl = F.create_map(*kv)

    # tipo de dato de la columna
    dtype = next(f.dataType for f in df.schema.fields if f.name == real_col)
    is_string = isinstance(dtype, T.StringType)

    if is_string:
        key_expr = F.upper(F.trim(F.col(real_col))) if normalize else F.col(real_col)
        new_value = F.coalesce(repl[key_expr], F.col(real_col))   # << uso de índice []
        return df.withColumn(real_col, new_value)

    if coerce_non_string:
        key_expr = (F.upper(F.trim(F.col(real_col).cast("string")))
                    if normalize else F.col(real_col).cast("string"))
        new_value = F.coalesce(repl[key_expr], F.col(real_col).cast("string")).cast(dtype)
        return df.withColumn(real_col, new_value)

    if verbose:
        print(f"[apply_company_code_fix] '{real_col}' es {dtype}. Omitido (coerce_non_string=False).")
    return df

In [ ]:
# 4-
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import expr
from delta.tables import DeltaTable
import json, sys

# Helper para armar condición de MERGE null-safe y opcionalmente normalizada
NULL_SAFE_KEYS = {"company_code"}
NORMALIZE_KEYS = {"company_code"}

def build_merge_condition(primary_keys: str,
                          null_safe_keys=NULL_SAFE_KEYS,
                          normalize_keys=NORMALIZE_KEYS,
                          use_coalesce=True) -> str:
    conds = []
    for key in [k.strip() for k in primary_keys.split(",")]:
        t = f"target.{key}"
        s = f"source.{key}"

        # normalización opcional
        if key in normalize_keys:
            t = f"upper(trim({t}))"
            s = f"upper(trim({s}))"

        if key in null_safe_keys:
            if use_coalesce:
                conds.append(f"COALESCE({t}, '') = COALESCE({s}, '')")
            else:
                conds.append(f"{t} <=> {s}")  # operador null-safe Spark SQL
        else:
            conds.append(f"{t} = {s}")
    return " AND ".join(conds)

def ensure_db_exists(full_table_name: str):
    """
    Soporta:
      - db.table
      - catalog.db.table (si aplica)
    """
    parts = full_table_name.split(".")
    try:
        if len(parts) == 2:
            db = parts[0]
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
        elif len(parts) == 3:
            catalog_db = ".".join(parts[:2])
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog_db}")
    except Exception as e:
        # Si el motor no soporta CREATE DATABASE con ese formato, no detenemos la ejecución
        print(f"⚠️ No se pudo asegurar la BD para '{full_table_name}'. Detalle: {str(e)}")

try:
    # ======================================================================
    # 0) VALIDAR TABLA DESTINO (SILVER) Y TIPO (DELTA / NO DELTA)
    # ======================================================================
    print(f"🔍 Validando existencia de la tabla destino '{target_table_name}'...")
    target_exists = spark.catalog.tableExists(target_table_name)
    target_is_delta = False

    if target_exists:
        # Si la tabla existe, intentamos abrirla como Delta.
        # Si truena, asumimos que NO es Delta y la recreamos.
        try:
            DeltaTable.forName(spark, target_table_name)
            target_is_delta = True
            print(f"✅ La tabla '{target_table_name}' existe y es una tabla Delta.")
        except Exception as e:
            print(f"⚠️ La tabla '{target_table_name}' existe pero NO es Delta.")
            print(f"   Detalle: {str(e)}")
            print("⚠️ Se eliminará del catálogo para recrearla como Delta.")
            spark.sql(f"DROP TABLE IF EXISTS {target_table_name}")
            target_exists = False
            target_is_delta = False
    else:
        print(f"ℹ️ La tabla '{target_table_name}' no existe todavía en el catálogo.")

    # ======================================================================
    # 1) CONTROL: leer último watermark / última ejecución exitosa
    # ======================================================================
    print("📥 Obteniendo el último watermark desde la tabla de control...")
    control_df = spark.sql(f"""
        SELECT last_watermark_value, date_sub(last_success_run_at, 6) AS last_success_run_at
        FROM {control_table_name}
        WHERE source_layer = '{source_layer}' AND
              source_table = '{source_table}' AND
              target_layer = '{target_layer}' AND
              target_table = '{target_table}'
    """)
    if control_df.count() > 0:
        row = control_df.first()
        last_watermark_value = row['last_watermark_value']
        last_success_run_at = row['last_success_run_at']
        print(f"Último watermark de '{watermark_column}': {last_watermark_value}")
        print(f"Última ejecución exitosa: {last_success_run_at}")
    else:
        print("⚠️ No se encontró registro en la tabla de control. Se considera como primera carga.")
        last_watermark_value = None
        last_success_run_at = None

    # ======================================================================
    # 2) ELIMINACIONES desde UPDT_LOG (si aplica y la tabla Silver YA ES Delta)
    # ======================================================================
    if updated_at_column == 'UPDT_LOG' and last_success_run_at is not None:
        if target_is_delta:
            print("🗑️ Buscando registros para eliminar en la capa Silver...")
            df_updt_log = spark.table(audit_table_name)
            df_delete_keys = (df_updt_log
                .filter(
                    (F.col("UPDATED_AT") >= last_success_run_at) &
                    (F.col("TABLE_NAME") == source_table) &
                    (F.col("OPERATION_TYPE") == 'D')
                )
                .select(
                    F.col("RECORD_ID").alias("RECORD_ID"),
                    F.upper(F.trim(F.col("COMPANY_CODE"))).alias("COMPANY_CODE")
                )
                .distinct()
            )
            df_delete_keys.cache()
            count_deletes = df_delete_keys.count()
            if count_deletes > 0:
                print(f"Se encontraron {count_deletes} registros para eliminar. Ejecutando MERGE de eliminación...")
                pk_list = [key.strip() for key in primary_keys.split(',')]
                merge_delete_condition = f"""
                    target.{pk_list[0]} = deletes.RECORD_ID
                    AND COALESCE(upper(trim(target.{pk_list[1]})), '') = COALESCE(deletes.COMPANY_CODE, '')
                """

                def _do_delete_merge():
                    t = DeltaTable.forName(spark, target_table_name)
                    (t.alias("target").merge(df_delete_keys.alias("deletes"), merge_delete_condition).whenMatchedDelete().execute())

                run_delta_operation_with_retry(_do_delete_merge, "MERGE de eliminación")
                print("✅ Eliminaciones completadas en la capa Silver.")
            else:
                print("No se encontraron nuevos registros para eliminar.")
            df_delete_keys.unpersist()
        else:
            print("ℹ️ Saltando eliminaciones desde UPDT_LOG porque la tabla destino no existe o no es Delta.")

    # ======================================================================
    # 3) INCREMENTAL: watermark / fecha modificación
    #     - Si NO hay watermark o NO existe Silver → FULL desde Bronze
    # ======================================================================
    print("Aplicando lógica de carga incremental basada en watermark y fecha de modificación...")

    # Si no hay watermark o la tabla Silver no existe (o la tratamos como tal),
    # ignoramos el watermark y traemos TODO desde Bronze.
    if last_watermark_value is None or not target_exists:
        print("⚠️ No hay watermark registrado o la tabla Silver no existe.")
        print("⚠️ Se realizará carga COMPLETA desde Bronze (ignorando watermark).")
        df_bronze_incremental = spark.table(source_table_name)
    else:
        # Carga incremental normal basada en watermark y/o columna de actualización
        filtro_watermark = F.col(watermark_column) > F.lit(last_watermark_value)
        if updated_at_column is not None and updated_at_column != 'UPDT_LOG' and last_success_run_at is not None:
            filtro_fecha = F.col(updated_at_column) >= F.lit(last_success_run_at).cast("date")
            df_bronze_incremental = spark.table(source_table_name).filter(filtro_watermark | filtro_fecha)
        else:
            df_bronze_incremental = spark.table(source_table_name).filter(filtro_watermark)

    # ======================================================================
    # 3.1) Agregar cambios I/U desde UPDT_LOG (si aplica)
    # ======================================================================
    if updated_at_column == 'UPDT_LOG' and last_success_run_at is not None:
        print("Aplicando lógica adicional de UPDT_LOG para 'I' y 'U'...")
        df_updt_log = spark.table(audit_table_name)
        df_changed_keys = (df_updt_log
            .filter(
                (F.col("UPDATED_AT") >= last_success_run_at) &
                (F.col("TABLE_NAME") == source_table) &
                (F.col("OPERATION_TYPE").isin('I', 'U'))
            )
            .select("RECORD_ID", "COMPANY_CODE").distinct()
        )
        df_changed_keys.cache()
        if df_changed_keys.rdd.isEmpty():
            print("ℹ️  No se encontraron inserciones o actualizaciones adicionales en UPDT_LOG.")
            df_changed_keys.unpersist()
        else:
            df_source_for_join = spark.table(source_table_name)
            domain_column_name = f"{source_table.split('_')[0]}_DOMAIN"
            df_updt_log_records = (df_source_for_join.alias("A").join(
                    df_changed_keys.alias("B"),
                    (F.col("A.PROGRESS_RECID") == F.col("B.RECORD_ID")) &
                    (F.col(f"A.{domain_column_name}") == F.col("B.COMPANY_CODE")),
                    "inner"
                ).select("A.*")
            )
            df_changed_keys.unpersist()
            print("🔗 Fusionando registros de watermark/fecha con UPDT_LOG...")
            df_bronze_incremental = df_bronze_incremental.union(df_updt_log_records).distinct()
            print(f"✅ Total de registros después de fusionar: {df_bronze_incremental.count()}")

    # ======================================================================
    # 3.2) Validación final incremental
    # ======================================================================
    if df_bronze_incremental.rdd.isEmpty():
        print("✅ No se encontraron registros nuevos o actualizados. Finalizando ejecución.")
        run_control_update(f"""
            UPDATE {control_table_name}
            SET last_run_status = 'Success',
                last_run_at = current_timestamp(),
                last_success_run_at = current_timestamp(),
                last_message = 'Load successful, no new data found.'
            WHERE source_layer = '{source_layer}' AND source_table = '{source_table}' AND
                target_layer = '{target_layer}' AND target_table = '{target_table}';
        """)
        notebookutils.notebook.exit("No new data to process.")
    else:
        df_bronze_incremental.cache()
        print(f"📊 Se encontraron {df_bronze_incremental.count()} registros para insertar/actualizar.")

        # ==================================================================
        # 4) TRANSFORMACIONES según mapping
        # ==================================================================
        print("✨ Aplicando transformaciones de esquema...")
        column_mappings = json.loads(column_mapping_json)
        select_expressions = []
        for m in column_mappings:
            source_expr_str = m['source']
            target_name = m['target']
            cast_type = m['cast']
            transform_type = m.get('transform')
            column_expression = expr(source_expr_str).cast(cast_type)
            if transform_type == 'title_case':
                final_expression = custom_title_case_udf(column_expression).alias(target_name)
            else:
                final_expression = column_expression.alias(target_name)
            select_expressions.append(final_expression)

        #df_source_transformed = df_bronze_incremental.select(*select_expressions)
        df_source_transformed = df_bronze_incremental.select(*select_expressions).withColumn("source_system", F.lit("QAD"))

        # 4.1) Normalización y fix de llaves para MERGE
        df_source_transformed = (df_source_transformed
            .withColumn("record_id", F.col("record_id").cast("long"))
            .withColumn("company_code", F.upper(F.trim(F.col("company_code"))))
        )
        # NO HABILITAR ESTA PARTE (FUNCION apply_company_code_fix )
        # df_source_transformed = apply_company_code_fix(df_source_transformed, col_name="company_code")

        # 4.2) Deduplicación por llave, conservando la fila con mayor last_updated_at
        w = Window.partitionBy("record_id", "company_code").orderBy(F.col("last_updated_at").desc_nulls_last())
        df_source_transformed = (df_source_transformed
            .withColumn("rn", F.row_number().over(w))
            .filter(F.col("rn") == 1)
            .drop("rn")
        )

        # ==================================================================
        # 5) MERGE (upsert) o creación de tabla Silver
        # ==================================================================
        # NOTA:
        # - Si la tabla NO existe o la recreamos → se crea/recrea con overwrite.
        # - Si sí existe y es Delta → se hace MERGE incremental normal.
        if not target_exists or not target_is_delta:
            print(f"ℹ️ La tabla '{target_table_name}' no existe como Delta (o fue recreada).")
            ensure_db_exists(target_table_name)

            (df_source_transformed.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .option("mergeSchema", "true")
                .saveAsTable(target_table_name))

            print("✅ Carga inicial / recreación de tabla Silver completada.")
        else:
            print(f"⚙️ Ejecutando MERGE para inserciones y actualizaciones en '{target_table_name}'...")

            def _do_upsert_merge():
                t = DeltaTable.forName(spark, target_table_name)
                mc = build_merge_condition(primary_keys)
                (t.alias("target").merge(df_source_transformed.alias("source"), mc).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

            run_delta_operation_with_retry(_do_upsert_merge, "MERGE de inserciones/actualizaciones")
            print("✅ MERGE de inserciones/actualizaciones completado.")

    # ======================================================================
    # 6) CONTROL: actualizar watermark/estado
    # ======================================================================
    print("📈 Actualizando la tabla de control...")
    new_max_wm = None
    if not df_bronze_incremental.rdd.isEmpty():
        new_max_wm = df_bronze_incremental.agg(F.max(F.col(watermark_column)).cast("long")).first()[0]

    update_wm_clause = f"last_watermark_value = '{new_max_wm}'," if new_max_wm is not None else ""

    run_control_update(f"""
        UPDATE {control_table_name}
        SET {update_wm_clause}
            last_run_status = 'Success',
            last_run_at = current_timestamp(),
            last_success_run_at = current_timestamp(),
            last_message = 'Load successful'
        WHERE source_layer = '{source_layer}' AND
              source_table = '{source_table}' AND
              target_layer = '{target_layer}' AND
              target_table = '{target_table}';
    """)
    if new_max_wm is not None:
        print(f"📝 Tabla de control actualizada. Nuevo watermark: {new_max_wm}")
    else:
        print("📝 Tabla de control actualizada (estado y fecha, sin nuevo watermark).")

    if not df_bronze_incremental.rdd.isEmpty():
        df_bronze_incremental.unpersist()

    print("🎉 Proceso finalizado con éxito.")

except Exception as e:
    print(f"!!!!!! ERROR en la carga de '{target_table_name}': {str(e)} !!!!!!", file=sys.stderr)
    err_msg = str(e).replace("'", "''")
    run_control_update(f"""
        UPDATE {control_table_name}
        SET last_run_status = 'Failed',
            last_run_at = current_timestamp(),
            last_message = '{err_msg}'
        WHERE source_layer = '{source_layer}' AND
              source_table = '{source_table}' AND
              target_layer = '{target_layer}' AND
              target_table = '{target_table}';
    """)
    raise e
